In [4]:
import random
import statistics

# Eight possible movement directions
DIRS = [
    (-1, 0), (1, 0), (0, -1), (0, 1),
    (-1, -1), (-1, 1), (1, -1), (1, 1)
]

In [5]:
class Courtyard:
    """Students move on a wrapped grid and spread a rumour."""

    def __init__(self, n=100, size=30, p=0.3, seed=0):
        self.rng = random.Random(seed)
        self.n = n
        self.size = size
        self.p = p
        self.tick = 0

        # Random initial positions
        self.pos = [
            (self.rng.randrange(size), self.rng.randrange(size))
            for _ in range(n)
        ]

        # Student 0 starts knowing the rumour
        self.knows = [False] * n
        self.knows[0] = True

    def count(self):
        return sum(self.knows)

    def render(self):
        """K = knows, s = unaware, . = empty."""
        rows = [["."] * self.size for _ in range(self.size)]

        for i, (r, c) in enumerate(self.pos):
            if self.knows[i]:
                rows[r][c] = "K"
            elif rows[r][c] == ".":
                rows[r][c] = "s"

        return "\n".join("".join(row) for row in rows)

    def move_all(self):
        for i in range(self.n):
            dr, dc = self.rng.choice(DIRS)
            r, c = self.pos[i]

            self.pos[i] = (
                (r + dr) % self.size,
                (c + dc) % self.size
            )

    def next_to_knower(self, cell, know_cells):
        r, c = cell

        return any(
            ((r + a) % self.size, (c + b) % self.size)
            in know_cells
            for a in (-1, 0, 1)
            for b in (-1, 0, 1)
        )

    def spread(self):
        know_cells = {
            self.pos[i]
            for i in range(self.n)
            if self.knows[i]
        }

        newly = [
            i for i in range(self.n)
            if not self.knows[i]
            and self.next_to_knower(self.pos[i], know_cells)
            and self.rng.random() < self.p
        ]

        for i in newly:
            self.knows[i] = True

    def step(self):
        self.move_all()
        self.spread()
        self.tick += 1

In [6]:
def ticks_until(model, fraction, max_ticks=600):
    target = fraction * model.n

    while model.count() < target and model.tick < max_ticks:
        model.step()

    return model.tick


def average_ticks(p, fraction, seeds=range(30), **kwargs):
    return statistics.mean(
        ticks_until(
            Courtyard(p=p, seed=s, **kwargs),
            fraction
        )
        for s in seeds
    )

In [7]:
class CourtyardStayHome(Courtyard):

    def __init__(self, stay_home=0.0, **kwargs):
        super().__init__(**kwargs)

        # Calculate the number of students staying home
        k = round(stay_home * self.n)

        # Select the last k student IDs
        self.frozen = set(range(self.n - k, self.n))

    def move_all(self):
        # Move students except those who stay home
        for i in range(self.n):

            if i in self.frozen:
                continue

            dr, dc = self.rng.choice(DIRS)
            r, c = self.pos[i]

            self.pos[i] = (
                (r + dr) % self.size,
                (c + dc) % self.size
            )

In [8]:
for stay_home in [0.0, 0.3, 0.6]:

    # Average ticks until 50% know the rumour
    half = statistics.mean(
        ticks_until(
            CourtyardStayHome(stay_home=stay_home, seed=s),
            0.5
        )
        for s in range(30)
    )

    # Average ticks until 90% know the rumour
    ninety = statistics.mean(
        ticks_until(
            CourtyardStayHome(stay_home=stay_home, seed=s),
            0.9
        )
        for s in range(30)
    )

    # Count runs where everyone knows within 600 ticks
    everyone = 0

    for s in range(30):
        m = CourtyardStayHome(stay_home=stay_home, seed=s)
        ticks_until(m, 1.0)
        everyone += (m.count() == m.n)

    print(
        f"stay home {stay_home}: half {half:.1f}, "
        f"90% {ninety:.1f}, "
        f"everyone knows in {everyone} of 30 runs"
    )

stay home 0.0: half 43.8, 90% 64.4, everyone knows in 30 of 30 runs
stay home 0.3: half 50.6, 90% 76.4, everyone knows in 30 of 30 runs
stay home 0.6: half 71.8, 90% 112.1, everyone knows in 30 of 30 runs
